# Full VNA sweep: find and fit resonances

Four steps, all saved in one zarr group:

1. **Auto finder** (choose one): the interactive classical finder, or the
   CNN finder. Saves a starting list as `fres_auto`.
2. **Manual finder**: review and correct the list by hand. Saves
   `fres_manual`.
3. **Fitter**: fits every resonance in `fres_manual` and saves the results
   in `full_sweep_fit`.
4. **Sweep description**: where the resonances are (range, banks), their
   quality factors, and the quality of the sweep (noise, gain, phase,
   sampling, notes). Printed by the fitter too, and reprinted any time
   from the saved results.

The CNN finder needs PyTorch: use the `citkid-cnn` environment
(`mamba env create -f environment-cnn.yml`), or add it to an existing
environment with
`mamba install -n citkid -c conda-forge pytorch=2.13.0 numpy=1.26.4`.
Everything else works without it.

In [ ]:
import zarr
from citkid.vna import (run_res_finder_auto, run_res_finder_cnn,
                        run_res_finder_manual, run_full_sweep_fit,
                        run_sweep_info)

# Load data
# f is in Hz
# z is complex S21 data (I + jQ; the fitter needs the phase)
f, z = ...

# Zarr group for all outputs of this sweep
path = ''
zarr_grp = zarr.open(
    path,
    mode = 'a'
)

## Step 1: auto finder (run one of the two cells)

**Option A, classical (interactive):** adjust smoothing and dip-detection
parameters while watching the result; close the window to save
`fres_auto`. If `fres_auto` exists, a popup asks whether to overwrite it.

**Option B, CNN:** a trained network finds the resonances; each candidate
is then fit, and only those that look like resonances are kept
(`fit_candidates = False` keeps every candidate). The model is downloaded
once on first use. Runs in seconds to a few minutes (the fits take most
of the time; a GPU speeds up the network part). Pass `overwrite = True`
to replace an existing `fres_auto`.

In [ ]:
# Option A: classical auto finder (interactive window)
fres_auto = run_res_finder_auto(f, z, zarr_grp = zarr_grp)

In [ ]:
# Option B: CNN finder (needs PyTorch)
fres_auto = run_res_finder_cnn(f, z, zarr_grp)

## Step 2: manual finder

Starts from `fres_auto` and saves the reviewed list as `fres_manual`. If
`fres_manual` exists, a popup asks whether to load it, overwrite it
(starting from `fres_auto`), or cancel.

In [ ]:
fres_manual = run_res_finder_manual(
    f, z,
    fres_initial = zarr_grp,
    zarr_grp = zarr_grp
)

## Step 3: fit every resonance

Every frequency in `fres_manual` gets one result (in the same order);
none are added or removed. Close resonances are fit together. Results in
`zarr_grp['full_sweep_fit']`: `fr`, `Qr`, `Qi`, `Qc`, `amp`, `phi` and
their `_err`, `snr`, `fit_status` (`ok`, `stage_a`, or `failed: ...` with
only `fr` kept), `flags` (`overlap`: Qc, phi unreliable from neighbors'
reflections, fr and Qi fine; `poor fit` / `distorted`: only fr reliable;
`bifurcated`), and `gain/f`, `gain/gain` (the smooth gain; the
gain-removed sweep is z / gain). The printed description is also saved
(`attrs['report']`).

Options: `fit_nonlinear = True` also fits each resonator's nonlinearity
`a` (a above ~0.77 is flagged `bifurcated`); `downward = False` for sweeps
taken upward in frequency (only matters with `fit_nonlinear`);
`overwrite = True` to replace an earlier fit.

In [ ]:
res, report = run_full_sweep_fit(f, z, zarr_grp)

## Step 4: sweep description

Reads the saved fit (no refitting): the range and gaps, the bank layout
(for 600 or more resonances), the median quality factors of the reliable
fits and how they vary, either with frequency across the band, or with
position within each bank, repeating in every bank (rises or falls toward
the bank's high end, highest or lowest in the middle of the bank), the
points per linewidth and SNR, the baseline noise, gain range, ripple and
sharp gain features, phase jumps, and notes on anything that makes the
sweep less reliable (thresholds in
`citkid.vna.sweep_info.NOTE_THRESHOLDS`).

In [ ]:
report = run_sweep_info(zarr_grp)

In [ ]:
# The results (also in zarr_grp['full_sweep_fit'])
fit = zarr_grp['full_sweep_fit']
reliable = (fit['fit_status'][:] == 'ok') & (fit['flags'][:] == '')
fr, Qi, Qc = fit['fr'][:], fit['Qi'][:], fit['Qc'][:]
res[reliable]

## Manual finder and auto finder reference


**Stage 1: Automatic Peak Finding**
- Run `run_res_finder_auto()` to automatically detect resonances
- Adjustable parameters: smoothing, peak height/width/distance
- Real-time preview of detected peaks
- Saves initial guess as `fres_auto` in zarr

**Stage 2: Manual Curation**
- Run `run_res_finder_manual()` to review and correct
- Loads `fres_auto` as starting point
- Click to add, Shift+click to remove
- Fast keyboard navigation (Z/X/A/S keys)
- Saves final list as `fres_manual` in zarr

## Basic Workflow (Manual Finder)

1. **Run the finder** - opens interactive window with 3 plots
2. **Navigate** - pan through the sweep using keyboard shortcuts
3. **Edit resonances** - click to add, Shift+click to remove
4. **Close window** - automatically saves to zarr

**Window layout:**
- **Top:** Overview navigator (drag yellow region to jump around)
- **Middle:** Magnitude plot (|S21| in dB)
- **Bottom:** Phase plot (unwrapped, trend-removed)
- **Right (Q key):** IQ plot overlay (toggle with Q)

**Visual encoding:**
- Red vertical lines: resonances
- Yellow highlight: IQ window region (when IQ plot visible)

## Navigation (Left Hand Keyboard)

**Pan/Zoom:**
- Mouse wheel: zoom in/out
- Drag yellow region in overview: jump to that section
- `Z`/`X`: pan left/right 20% (fine adjustment)
- `A`/`S`: pan left/right 80% (jump between resonances)

**Pro tip:** Use A/S to quickly jump through ~10 resonances per keypress for fast review

## Adding and Removing Resonances

**Add a resonance:**
- **Left-click** on empty space in magnitude or phase plot
- Click location determines frequency
- Won't add if another resonance is within 1 data point

**Remove a resonance:**
- **Shift+Left-click** near the resonance you want to remove
- Removes the nearest resonance to click location
- Safe: requires exact Shift modifier to prevent accidents

**IQ plot mode (Q key):**
- Toggle IQ plot visible with `Q` key
- Yellow highlight shows IQ window around selected frequency
- Click in IQ plot to add resonance at nearest data point
- Shift+click in IQ plot to remove nearest resonance
- Useful for examining resonance circle shapes

## Undo

- `Ctrl+Z`: undo last action
- Undo stack remembers all adds and removes
- Keep pressing Ctrl+Z to step backward through changes

## Saving

**Auto-saves on close** - just close the window and data is written to zarr as `fres_manual`.

Or use:
- `S` key: save now (keeps window open)
- "Quit and Save" button: save and close

## Keyboard Reference

**Navigation:**
- `Z` / `X` - pan 20% left/right
- `A` / `S` - pan 80% left/right (fast jumps)
- Scroll wheel - zoom

**Editing:**
- Click - add resonance
- Shift+Click - remove nearest resonance
- `Ctrl+Z` - undo

**Other:**
- `Q` - toggle IQ plot overlay
- `S` - save to zarr
- `H` - show help dialog

## Auto Finder Parameters

When running the automatic finder first, you can adjust these parameters:

**Frequency Range:**
- `f_min` / `f_max` - restrict search to sub-band

**Smoothing:**
- `highpass` - removes baseline drift (good for long sweeps)
- `polynomial` - fits polynomial baseline (good for smooth backgrounds)
- `none` - no smoothing (for pre-processed data)

**Peak Detection:**
- `height` - minimum dip depth in dB (default: 5 dB below baseline)
- `width` - minimum peak width in kHz (prevents noise spikes)
- `distance` - minimum spacing between peaks in kHz (prevents duplicates)
---